# DALIL : modèle de langage affiné par LoRA (variante D, section 6.6)

Paramètres Kaggle : **GPU T4** (ou P100), **Internet activé**. Entrée : `data/processed/llm/` produit par `python ml/build_llm_dataset.py` (relancer ce script après la traduction pour profiter du corpus). Sortie : fichier GGUF servi sur le portable par llama.cpp (`ml/kaggle/README.md`).

Gemma 3 demande d'accepter sa licence sur Hugging Face et un jeton `HF_TOKEN` dans les secrets Kaggle.

In [ ]:
import glob, json, os, random, shutil, time
from collections import Counter
# Le jeu de données « dalil-pipeline » est monté sous /kaggle/input (chemin variable selon Kaggle) :
# on cherche le dossier qui contient backend/ et data/.
INPUT = next(root for root, dirs, _ in os.walk("/kaggle/input") if "backend" in dirs and "data" in dirs)
REPO = "/kaggle/working/dalil"
if not os.path.exists(REPO):
    shutil.copytree(INPUT, REPO)
os.chdir(REPO)
print(sorted(os.listdir(REPO)))

In [ ]:
!pip install -q -U "transformers>=4.51" peft trl bitsandbytes accelerate datasets
!pip uninstall -y -q torchao

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"   # à comparer : "google/gemma-3-4b-it", "Qwen/Qwen3-8B" (variante plus lourde)
NAME = "dalil-" + MODEL_ID.split("/")[-1].lower()
OUT = f"/kaggle/working/{NAME}"
ADAPTER = f"{OUT}/adapter"
EPOCHS, LR, MAX_LEN = 1, 2e-4, 2048
N_EVAL = 100   # exemples de validation générés à la fin (la génération est lente)
MAX_TRAIN_H, MAX_EVAL_MIN = 3.5, 30   # garde-fous : la session Kaggle s'arrête à 12 h, on garde de la marge

In [ ]:
# Un seul GPU visible, avant le premier import de torch : SFTTrainer refuse nn.DataParallel (T4 x2).
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainerCallback
from trl import SFTConfig, SFTTrainer

data = load_dataset("json", data_files={"train": "data/processed/llm/train.jsonl", "val": "data/processed/llm/val.jsonl"})
tok = AutoTokenizer.from_pretrained(MODEL_ID)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": 0}, dtype=torch.float16)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
# Les GPU T4 et P100 ne gèrent pas le bfloat16. TRL active bf16 par défaut : on le coupe explicitement et on
# entraîne en précision mixte float16, adaptateurs LoRA en float32 (sinon le GradScaler échoue).
# Un seul GPU (device_map {"": 0}) : réparti sur deux T4, le modèle passe son temps à attendre l'autre carte.
# Pas de prepare_model_for_kbit_training : il passe embeddings et tête de sortie en float32, ce qui divise
# la vitesse par plusieurs fois sur T4.
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model = get_peft_model(model, lora)
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()
model.print_trainable_parameters()
cfg = SFTConfig(output_dir=OUT, num_train_epochs=EPOCHS, learning_rate=LR, per_device_train_batch_size=2,
                gradient_accumulation_steps=8, gradient_checkpointing=True,
                gradient_checkpointing_kwargs={"use_reentrant": False}, fp16=True, bf16=False,
                logging_steps=5, logging_first_step=True, disable_tqdm=True,   # journal lisible en direct sur Kaggle
                save_strategy="steps", save_steps=40, save_total_limit=1, eval_strategy="no",
                max_length=MAX_LEN, report_to="none")
class TimeBudget(TrainerCallback):
    """Arrête proprement l'entraînement après MAX_TRAIN_H heures : l'adaptateur est sauvegardé, puis
    l'évaluation et la conversion GGUF tournent quand même. Évite une nuit perdue si le GPU est lent."""
    def on_train_begin(self, args, state, control, **kw):
        self.t0 = time.time()
    def on_step_end(self, args, state, control, **kw):
        if time.time() - self.t0 > MAX_TRAIN_H * 3600:
            print(f"Budget de temps atteint à l'étape {state.global_step}/{state.max_steps} : arrêt de l'entraînement", flush=True)
            control.should_training_stop = True
        return control

trainer = SFTTrainer(model=model, args=cfg, train_dataset=data["train"], eval_dataset=data["val"],
                     processing_class=tok, callbacks=[TimeBudget()])
# SFTTrainer repasse les adaptateurs d'un modèle quantifié en bfloat16 : le GradScaler float16 échoue alors
# (« _amp_foreach_non_finite_check_and_unscale_cuda not implemented for BFloat16 »). On les remet en float32
# après sa construction ; l'optimiseur n'est créé qu'au lancement de train().
for param in trainer.model.parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.float()
print("types des paramètres entraînés :", Counter(str(p.dtype) for p in trainer.model.parameters() if p.requires_grad))
resume = bool(glob.glob(f"{OUT}/checkpoint-*"))   # reprise après une session interrompue
trainer.train(resume_from_checkpoint=resume)
trainer.save_model(f"{OUT}/adapter")

In [ ]:
# Passage en mode génération : sans cela le gradient checkpointing laisse le cache désactivé
# (~50 s par réponse) et le dropout LoRA reste actif.
model.gradient_checkpointing_disable(); model.config.use_cache = True; model.eval()

In [ ]:
# Évaluation sur la validation : JSON valide, panne correcte en tête, aucune panne hors des candidats
import re
def generate(messages, max_new_tokens=512):
    prompt = tok.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

ok_json = ok_top1 = ok_cands = 0
samples = []
t_eval = time.time()
val = data["val"].select(range(min(N_EVAL, len(data["val"]))))
for i, ex in enumerate(val):
    gold = json.loads(ex["messages"][2]["content"])["ranking"][0]["fault"]
    cands = {c["slug"] for c in json.loads(ex["messages"][1]["content"])["candidats"]}
    text = generate(ex["messages"])
    if i < 3:
        print(f"--- réponse {i} (attendu : {gold}) ---\n{text[:800]}", flush=True)
    m = re.search(r"\{.*\}", text, re.S)
    try:
        ranking = json.loads(m.group(0))["ranking"]
        ok_json += 1
        ok_top1 += ranking[0]["fault"] == gold
        ok_cands += all(r["fault"] in cands for r in ranking)
    except Exception:
        pass
    samples.append({"gold": gold, "output": text})
    if time.time() - t_eval > MAX_EVAL_MIN * 60:
        print(f"Budget d'évaluation atteint après {i + 1} exemples", flush=True)
        val = val.select(range(i + 1))
        break
    if i % 10 == 0:
        print(f"évaluation {i}/{len(val)} ({time.time() - t_eval:.0f} s)", flush=True)
n = len(val)
result = {"n": n, "json_valide": ok_json, "top1": ok_top1, "candidats_respectes": ok_cands}
print(f"JSON valide {ok_json}/{n}, panne correcte en tête {ok_top1}/{n}, candidats respectés {ok_cands}/{n}")
with open(f"/kaggle/working/{NAME}-evaluation.json", "w", encoding="utf-8") as fh:
    json.dump({"resultat": result, "reponses": samples}, fh, ensure_ascii=False, indent=1)

In [ ]:
del model, trainer; torch.cuda.empty_cache()

In [ ]:
# Fusion de l'adaptateur dans le modèle de base (float16, sur processeur)
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="cpu")
merged = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()
merged.save_pretrained(f"/tmp/merged", safe_serialization=True)
del base, merged
# Fichiers du tokenizer d'origine : ceux réécrits par transformers 5 font échouer convert_hf_to_gguf
# (« 'list' object has no attribute 'keys' »).
from huggingface_hub import snapshot_download
src = snapshot_download(MODEL_ID, allow_patterns=["tokenizer*", "vocab.json", "merges.txt", "special_tokens_map.json",
                                                  "generation_config.json"])
for f in os.listdir(src):
    if os.path.isfile(os.path.join(src, f)):
        shutil.copy(os.path.join(src, f), f"/tmp/merged/{f}")
print(sorted(os.listdir(f"/tmp/merged")))

In [ ]:
# Conversion en GGUF pour llama.cpp (processeur du portable)
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp
!pip install -q -r /kaggle/working/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!pip uninstall -y -q torchao
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py /tmp/merged --outfile /tmp/{NAME}-f16.gguf --outtype f16
!rm -rf /tmp/merged && df -h /tmp /kaggle/working
# llama-quantize refuse de recompresser un fichier déjà en q8_0 : on part du f16. Modèle fusionné et f16 (8 Go
# chacun) dans /tmp : /kaggle/working est limité à ~20 Go (« Not enough free space »).
!cd /kaggle/working/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build build --target llama-quantize -j 4 > /dev/null
!/kaggle/working/llama.cpp/build/bin/llama-quantize /tmp/{NAME}-f16.gguf /kaggle/working/{NAME}-q4_k_m.gguf Q4_K_M
!/kaggle/working/llama.cpp/build/bin/llama-quantize /tmp/{NAME}-f16.gguf /kaggle/working/{NAME}-q8_0.gguf Q8_0
!rm -f /tmp/{NAME}-f16.gguf
!rm -rf /kaggle/working/llama.cpp && ls -lh /kaggle/working/*.gguf